In [1]:
import sys
!{sys.executable} -m pip install langchain-experimental

  Using cached langchain_experimental-0.4.0-py3-none-any.whl.metadata (1.3 kB)
Using cached langchain_experimental-0.4.0-py3-none-any.whl (209 kB)


In [1]:
import json
from pathlib import Path
from langchain_experimental.text_splitter import SemanticChunker
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

c:\KINO\RAG-Project\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Настройка путей
PROJECT_ROOT = Path.cwd().parent 
processed_path = PROJECT_ROOT / "data" / "processed" / "deep_learning_goodfellow.txt"
faiss_index_path = str(PROJECT_ROOT / "faiss_index")
BOOK_NAME = "deep_learning_goodfellow.pdf"

In [3]:
# Загрузка текста из книги
with open(processed_path, "r", encoding="utf-8") as f:
    raw_text = f.read()

In [4]:
embedding_model = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")

C:\Users\natas\AppData\Local\Temp\ipykernel_57840\1575338406.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name="multi-qa-mpnet-base-dot-v1")


SemanticChunker делит текст, основываясь на смысловой близости предложений. Разделяет текст там, где происходит смена темы или контекста. Он старается сгруппировать в один чанк предложения, которые говорят об одном и том же, и начать новый чанк, когда тема разговора меняется.
Представьте, что вы читаете книгу, SemanticChunker постарается закончить чанк в конце абзаца или смыслового блока.

- Разбиение на предложения

Сначала SemanticChunker берет текст и разбивает его на отдельные предложения. Он использует стандартные методы для определения границ предложений (например, по точкам, вопросительным знакам и т.д.).

- Создание эмбеддингов для каждого предложения

SemanticChunker берет модель эмбеддингов, которую вы ему передаете (например, HuggingFaceEmbeddings), и превращает каждое отдельное предложение в вектор — числовое представление его смысла.

- Расчет семантического сходства между соседними предложениями

Теперь, имея вектор для каждого предложения, чанкер начинает сравнивать их попарно. Он вычисляет косинусное сходство (cosine similarity) между вектором предложения 1 и предложения 2, затем между 2 и 3, 3 и 4, и так далее.
Косинусное сходство — это метрика, которая показывает, насколько "похожи" два вектора.

Значение близко к 1: предложения очень похожи по смыслу (векторы смотрят в одном направлении).

Значение близко к 0: предложения не связаны по смыслу.

Значение близко к -1: предложения имеют противоположный смысл.

- Определение "точек разрыва" (Breakpoints)

Чанкер анализирует все полученные значения сходства. Если сходство между двумя соседними предложениями резко падает, он помечает это место как потенциальную "точку разрыва".

Как он решает, что падение "резкое"? Он не использует фиксированное число (например, 0.5). Вместо этого он использует статистический подход, чаще всего процентиль (percentile). Например, он может определить "все точки, где сходство ниже 25-го процентиля" как места для разрыва. Это делает его адаптивным к разным текстам.

- Группировка предложений в чанки

Наконец, SemanticChunker берет все предложения между двумя точками разрыва и объединяет их в один смысловой чанк.

In [5]:
# Создаем семантический сплиттер
semantic_text_splitter = SemanticChunker(
    embedding_model,
    breakpoint_threshold_type="standard_deviation",
     breakpoint_threshold_amount=0.7
)

In [6]:
documents = semantic_text_splitter.create_documents([raw_text])

# Извлекаем тексты и формируем метаданные
texts = []
metas = []
for i, doc in enumerate(documents):
    texts.append(doc.page_content)
    metas.append({
        "chunk_id": i,
        "source": BOOK_NAME,
    })

print(f"Книга успешно разбита на {len(texts)} семантических чанков.")

Книга успешно разбита на 3615 семантических чанков.


In [7]:
# Создаём эмбеддинги для новых чанков и построим индекс
db = FAISS.from_texts(texts, embedding_model, metadatas=metas)

db.save_local(faiss_index_path)

print(f"Новая база FAISS создана.")
print(f"Она сохранена в папку: {faiss_index_path}")

Новая база FAISS создана.
Она сохранена в папку: c:\KINO\RAG-Project\faiss_index
